<a href="https://colab.research.google.com/github/isi1993/DRRR/blob/main/customer_behaviour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandasql

In [ ]:
import pandas as pd
import sqlite3
from pandasql import sqldf
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')



In [ ]:
pysqldf = lambda q: sqldf(q, globals())


In [ ]:
df = pd.read_csv('/content/customer_shopping_behavior (1).csv')
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [ ]:
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [ ]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'}, inplace=True)

In [ ]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [ ]:
# create a new columns age group
labels = ['Young Adults', 'Adult', 'Middle-Aged' , 'Senior Citizens']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [ ]:
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle-Aged
1,19,Young Adults
2,50,Middle-Aged
3,21,Young Adults
4,45,Middle-Aged
5,46,Middle-Aged
6,63,Senior Citizens
7,27,Young Adults
8,26,Young Adults
9,57,Middle-Aged


In [ ]:
# create column purchase frequency days
frequency_mapping = {
  'Fortnightly' : 14,
  'Weekly' : 7,
  'Monthly' : 30,
  'Quarterly' : 90,
  'Bi-Weekly' : 14,
  'Annually' : 365,
  'Every 3 months' : 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14.0,Fortnightly
1,14.0,Fortnightly
2,7.0,Weekly
3,7.0,Weekly
4,365.0,Annually
5,7.0,Weekly
6,90.0,Quarterly
7,7.0,Weekly
8,365.0,Annually
9,90.0,Quarterly


In [ ]:
df[['discount_applied', 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [ ]:
# check if both columns  above contain same value
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [ ]:
df.drop(columns=['promo_code_used'], axis=1, inplace=True)

In [ ]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [ ]:
#--Q1. What is the total revenue generated by male vs. female customers?
query = """
SELECT gender, SUM(purchase_amount) AS total_revenue
FROM df
GROUP BY gender
"""
sqldf(query)

,gender,total_revenue
0,Female,75191
1,Male,157890


In [ ]:
#--Q2. Which customers used a discount but still spent more than the average purchase amount?
query = """
SELECT customer_id, purchase_amount
FROM df
WHERE discount_applied = 'Yes' AND purchase_amount > (SELECT AVG(purchase_amount) FROM df)
"""
sqldf(query)


,customer_id,purchase_amount
0,2,64
1,3,73
2,4,90
3,7,85
4,9,97
...,...,...
834,1667,64
835,1671,73
836,1673,73
837,1674,62


In [ ]:
from pandasql import sqldf

In [ ]:
#--Q3. Which are the top 5 products with the highest average review rating?
query = """
SELECT item_purchased, ROUND(AVG(review_rating), 2) AS "Average Product Rating"
FROM df
GROUP BY item_purchased
ORDER BY AVG(review_rating) DESC
LIMIT 5
"""
sqldf(query)

,item_purchased,Average Product Rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,Skirt,3.78


In [ ]:
#--Q4. Compare the average Purchase Amounts between Standard and Express Shipping.
query = """
SELECT shipping_type, ROUND(AVG(purchase_amount), 2) as "Average Purchase Amount"
FROM df
where shipping_type in ('Standard', 'Express')
GROUP BY shipping_type
"""
sqldf(query)


,shipping_type,Average Purchase Amount
0,Express,60.48
1,Standard,58.46


In [ ]:
#--Q5. Do subscribed customers spend more? Compare average spend and total revenue between
query = """
SELECT subscription_status,
COUNT(customer_id) as "Total Customers",
ROUND(AVG(purchase_amount), 2) as Avg_spend,
ROUND(SUM(purchase_amount), 2) as  total_revenue
from df
GROUP BY subscription_status
order by total_revenue, avg_spend desc
"""
sqldf(query)

,subscription_status,Total Customers,Avg_spend,total_revenue
0,Yes,1053,59.49,62645.0
1,No,2847,59.87,170436.0


In [ ]:
# -06. Which 5 products have the highest percentage of purchases with discounts applied?
query = """
SELECT item_purchased,
ROUND(100 * SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END)/ COUNT(*), 2)AS discount_rate
FROM df
GROUP BY item_purchased
ORDER BY discount_rate DESC
LIMIT 5
"""
sqldf(query)


,item_purchased,discount_rate
0,Hat,50.0
1,Sneakers,49.0
2,Coat,49.0
3,Sweater,48.0
4,Pants,47.0


In [ ]:
query = """
WITH customer_type AS (
SELECT
  customer_id, previous_purchases,
  CASE
   WHEN previous_purchases = 1 THEN 'New'
   WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
   ELSE 'Loyal'
   END AS customer_segment
FROM df
)
SELECT
  customer_segment, COUNT(*) AS "Number of Customers"
  from customer_type
  GROUP BY customer_segment
"""
sqldf(query)

,customer_segment,Number of Customers
0,Loyal,3116
1,New,83
2,Returning,701


In [ ]:
#--08. What are the top 3 most purchased products within each category?
query = """
WITH item_counts AS (
  SELECT
    category,
    item_purchased,
    COUNT(customer_id) AS total_orders,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(customer_id) DESC) AS item_rank
    FROM df
  GROUP BY category, item_purchased
)
SELECT
item_rank,
  category,
  item_purchased,
  total_orders
FROM item_counts
WHERE item_rank <=3
"""
sqldf(query)


,item_rank,category,item_purchased,total_orders
0,1,Accessories,Jewelry,171
1,2,Accessories,Sunglasses,161
2,3,Accessories,Belt,161
3,1,Clothing,Pants,171
4,2,Clothing,Blouse,171
5,3,Clothing,Shirt,169
6,1,Footwear,Sandals,160
7,2,Footwear,Shoes,150
8,3,Footwear,Sneakers,145
9,1,Outerwear,Jacket,163


In [ ]:
#--Q9. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?
query = """
SELECT subscription_status,
COUNT(customer_id) as repeat_buyers
from df
WHERE previous_purchases > 5
GROUP BY subscription_status
"""
sqldf(query)

,subscription_status,repeat_buyers
0,No,2518
1,Yes,958


In [ ]:
#--Q10. What is the revenue contribution of each age group?
query = """
SELECT age_group, ROUND(SUM(purchase_amount), 2) AS "Total Revenue"
FROM df
GROUP BY age_group
order by SUM(purchase_amount) desc
"""
sqldf(query)


,age_group,Total Revenue
0,Young Adults,62143.0
1,Middle-Aged,59197.0
2,Adult,55978.0
3,Senior Citizens,55763.0


In [ ]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-Aged,14.0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adults,14.0
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-Aged,7.0
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adults,7.0
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-Aged,365.0


In [ ]:
df.to_csv('Customer_Shopping.csv', index=False)
from google.colab import files
files.download('Customer_Shopping.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>